# Notebook 4 — `inference.ipynb`

**Sovereign Dialect-Bridge · Step 4 — Evaluation + End-to-End Demo**

Evaluasi dua level:

| Level | Apa | Metrik |
|-------|-----|--------|
| **L1 per-stage** | Normalizer pada NusaX val | BLEU-4, chrF++ |
| **L1 per-stage** | Summarizer + baseline pada IndoSum test (700 stratified) | ROUGE-1/2/L, BERTScore-F1, CR |
| **L2 end-to-end** | Dialect test set (NusaX pseudo-articles, 3 dialek) | ROUGE without_norm vs with_norm, delta per dialek |

Output: `outputs/final_results.json`


## 1. Setup environment

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]   = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc, json, re, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = torch.cuda.is_bf16_supported() if DEVICE == "cuda" else False

print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM   : {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
print(f"bf16   : {USE_BF16}")


## 2. Install dependencies

```bash
pip install transformers==4.40.0 datasets accelerate sentencepiece
pip install rouge-score bert-score sacrebleu sacremoses
pip install PySastrawi networkx scikit-learn matplotlib
```


## 3. Paths & generation config

In [ ]:
CWD = Path.cwd()
ROOT        = CWD if (CWD / "data").exists() else CWD.parent
DATA_DIR    = ROOT / "data"
MODELS_DIR  = ROOT / "models"
OUTPUTS_DIR = ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

NORM_DIR     = MODELS_DIR / "normalizer"
INDOT5_DIR   = MODELS_DIR / "indot5"
INDOBART_DIR = MODELS_DIR / "indobart"
MT5BASE_DIR  = MODELS_DIR / "mt5base"

GEN_KWARGS = dict(
    max_new_tokens       = 150,
    num_beams            = 4,
    no_repeat_ngram_size = 3,
    early_stopping       = True,
    length_penalty       = 1.0,
)

print("Model checkpoints:")
for name, path in [("normalizer", NORM_DIR), ("indot5", INDOT5_DIR),
                    ("indobart",  INDOBART_DIR), ("mt5base", MT5BASE_DIR)]:
    status = "found" if path.exists() else "MISSING"
    print(f"  {name:12s} : {status}")


## 4. Load test data

In [ ]:
df_test = pd.read_parquet(DATA_DIR / "test.parquet")
print(f"Test set : {len(df_test):,} rows")
print(f"Categories: {df_test['category'].value_counts().to_dict()}")
print(f"Avg words : {df_test['word_count'].mean():.0f}")
df_test.head(3)


## 5. Preprocessing helpers

> Dua jalur terpisah: `preprocess_extractive` (dengan stem) untuk TextRank/NER; `preprocess_abstractive` (tanpa stem) untuk model neural. Jangan ditukar — stemming menurunkan ROUGE model neural 3-8 poin.


In [ ]:
try:
    from PySastrawi.Stemmer.StemmerFactory import StemmerFactory
except ImportError:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

_stemmer = StemmerFactory().create_stemmer()


def clean_noise(text: str) -> str:
    text = re.sub(r"https?://\S+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_case_punct(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\s.,!?;:-]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def stem_text(text: str) -> str:
    return " ".join(_stemmer.stem(w) for w in text.split())


def split_sentences(text: str) -> list:
    sents = re.split(r"(?<=[.!?])\s+", text.strip())
    return [s.strip() for s in sents if s.strip()]


def preprocess_extractive(text: str) -> str:
    return stem_text(normalize_case_punct(clean_noise(text)))


def preprocess_abstractive(text: str) -> str:
    # Tanpa stem — stemming menyebabkan distributional shift pada model neural
    return normalize_case_punct(clean_noise(text))


demo_preprocess = "Saya melaporkan jalan rusak!! https://t.co/xyz  Sudah lama tidak diperbaiki."
print("extractive :", preprocess_extractive(demo_preprocess))
print("abstractive:", preprocess_abstractive(demo_preprocess))


## 6. Baseline extractive — TextRank + NER

In [ ]:
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def summarize_textrank(text: str, n_sentences: int = 3, max_words: int = 80) -> str:
    sents = split_sentences(preprocess_extractive(text))
    if len(sents) <= n_sentences:
        result = " ".join(sents)
    else:
        try:
            vec    = TfidfVectorizer().fit_transform(sents)
            sim    = cosine_similarity(vec, vec)
            np.fill_diagonal(sim, 0)
            scores = nx.pagerank(nx.from_numpy_array(sim))
            ranked = sorted(scores, key=scores.get, reverse=True)[:n_sentences]
            result = " ".join(sents[i] for i in sorted(ranked))
        except Exception:
            result = " ".join(sents[:n_sentences])
    return " ".join(result.split()[:max_words])


def summarize_ner(text: str, n_sentences: int = 3, max_words: int = 80) -> str:
    sents = split_sentences(preprocess_extractive(text))
    if len(sents) <= n_sentences:
        result = " ".join(sents)
    else:
        def entity_score(s):
            tokens = s.split()
            caps   = sum(1 for t in tokens if t and t[0].isupper())
            return caps / max(len(tokens), 1)
        ranked = sorted(range(len(sents)),
                        key=lambda i: entity_score(sents[i]), reverse=True)
        result = " ".join(sents[i] for i in sorted(ranked[:n_sentences]))
    return " ".join(result.split()[:max_words])


## 7. Metric helpers

In [ ]:
from rouge_score import rouge_scorer as rs
from bert_score import score as bert_score_fn


def compute_rouge(preds, refs):
    scorer = rs.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
    scores = [scorer.score(r, p) for p, r in zip(preds, refs)]
    n = len(scores)
    return {
        "rouge1": round(sum(s["rouge1"].fmeasure for s in scores) / n, 4),
        "rouge2": round(sum(s["rouge2"].fmeasure for s in scores) / n, 4),
        "rougeL": round(sum(s["rougeL"].fmeasure for s in scores) / n, 4),
    }


def compute_bertscore(preds, refs, lang="id"):
    if not preds:
        return {"bertscore_f1": None}
    _, _, F1 = bert_score_fn(preds, refs, lang=lang, verbose=False, device=DEVICE)
    return {"bertscore_f1": round(F1.mean().item(), 4)}


def compute_cr(preds, originals):
    crs = [len(p.split()) / max(len(o.split()), 1)
           for p, o in zip(preds, originals)]
    return round(sum(crs) / len(crs), 4)


## 8. Summarizer inference helper

In [ ]:
def free_vram(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def generate_summaries(model_dir, model_type, df, max_input=512):
    """Load checkpoint, generate ringkasan untuk seluruh df, return list prediksi."""
    print(f"  Loading {Path(model_dir).name} ...")
    tok = AutoTokenizer.from_pretrained(
        str(model_dir), use_fast=False if model_type == "bart" else True
    )
    model = AutoModelForSeq2SeqLM.from_pretrained(str(model_dir)).to(DEVICE)
    model.eval()

    preds, t0 = [], time.time()
    for i, text in enumerate(df["text"].tolist()):
        text = preprocess_abstractive(text)
        if model_type == "mt5":
            text = "summarize: " + text
        inputs = tok(text, return_tensors="pt",
                     truncation=True, max_length=max_input).to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, **GEN_KWARGS)
        preds.append(tok.decode(out[0], skip_special_tokens=True))
        if (i + 1) % 100 == 0:
            print(f"    {i+1}/{len(df)}  elapsed={time.time()-t0:.0f}s")

    free_vram(model, tok)
    print(f"  done: {len(preds)} predictions  ({time.time()-t0:.0f}s)")
    return preds


## 9. Generate prediksi — 5 metode

Urutan: extractive dulu (CPU, tanpa load model), lalu neural satu per satu. Test set digunakan pertama kali di section ini.


In [ ]:
all_preds = {}

print("--- TextRank ---")
all_preds["TextRank"] = [
    summarize_textrank(t, n_sentences=3, max_words=80) for t in df_test["text"]
]
print(f"  {len(all_preds['TextRank'])} predictions")

print("\n--- NER ---")
all_preds["NER"] = [
    summarize_ner(t, n_sentences=3, max_words=80) for t in df_test["text"]
]
print(f"  {len(all_preds['NER'])} predictions")


In [ ]:
print("--- IndoT5 ---")
if INDOT5_DIR.exists():
    all_preds["IndoT5"] = generate_summaries(INDOT5_DIR, "t5", df_test)
else:
    all_preds["IndoT5"] = []
    print("  SKIP — models/indot5 tidak ditemukan")


In [ ]:
print("--- IndoBART ---")
if INDOBART_DIR.exists():
    all_preds["IndoBART"] = generate_summaries(INDOBART_DIR, "bart", df_test)
else:
    all_preds["IndoBART"] = []
    print("  SKIP — models/indobart tidak ditemukan")


In [ ]:
print("--- mT5-base ---")
if MT5BASE_DIR.exists():
    all_preds["mT5-base"] = generate_summaries(MT5BASE_DIR, "mt5", df_test)
else:
    all_preds["mT5-base"] = []
    print("  SKIP — models/mt5base tidak ditemukan")


## 10. Level 1 — metrik summarizer

In [ ]:
refs      = df_test["summary"].astype(str).tolist()
originals = df_test["text"].astype(str).tolist()

results_rows = []
for method, preds in all_preds.items():
    if not preds:
        continue
    print(f"  {method} ...")
    r   = compute_rouge(preds, refs)
    cr  = compute_cr(preds, originals)
    row = {
        "method": method,
        "rouge1": r["rouge1"], "rouge2": r["rouge2"], "rougeL": r["rougeL"],
        "cr": cr, "n": len(preds),
    }
    if method in {"IndoT5", "IndoBART", "mT5-base"}:
        row["bertscore_f1"] = compute_bertscore(preds, refs)["bertscore_f1"]
    else:
        row["bertscore_f1"] = None
    results_rows.append(row)
    bs = f"{row['bertscore_f1']:.4f}" if row["bertscore_f1"] is not None else "  —"
    print(f"    R1={row['rouge1']:.4f}  R2={row['rouge2']:.4f}  RL={row['rougeL']:.4f}"
          f"  BERTScore={bs}  CR={row['cr']:.4f}")


In [ ]:
methods_plot = [r["method"] for r in results_rows]
x, w = np.arange(len(methods_plot)), 0.25

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w, [r["rouge1"] for r in results_rows], w, label="ROUGE-1", color="steelblue")
ax.bar(x,     [r["rouge2"] for r in results_rows], w, label="ROUGE-2", color="darkorange")
ax.bar(x + w, [r["rougeL"] for r in results_rows], w, label="ROUGE-L", color="mediumseagreen")
ax.set_xticks(x); ax.set_xticklabels(methods_plot, fontsize=11)
ax.set_ylabel("F1 Score")
ax.set_title("ROUGE Comparison — 5 Methods on IndoSum Test")
ax.legend(); ax.set_ylim(0, 0.65)
plt.tight_layout(); plt.show()


## 11. Evaluasi Normalizer — BLEU-4 + chrF++

In [ ]:
import sacrebleu
from datasets import load_dataset

norm_metrics = {"bleu4": None, "chrf": None, "n_samples": 0}

if not NORM_DIR.exists():
    print("SKIP — models/normalizer tidak ditemukan")
else:
    nusax_val  = None
    local_path = ROOT / "dataset" / "nusax" / "datasets" / "mt" / "valid.csv"
    if local_path.exists():
        nusax_val = pd.read_csv(local_path)
        print(f"NusaX val (local): {len(nusax_val)} rows")
    else:
        try:
            ds = load_dataset("indonlp/NusaX-MT", trust_remote_code=True)
            split = "validation" if "validation" in ds else "valid"
            nusax_val = ds[split].to_pandas()
        except Exception as e:
            print(f"Gagal load NusaX-MT: {e}")

    if nusax_val is not None:
        norm_tok = AutoTokenizer.from_pretrained(str(NORM_DIR))
        norm_mod = AutoModelForSeq2SeqLM.from_pretrained(str(NORM_DIR)).to(DEVICE)
        norm_mod.eval()

        DIALECT_COLS = [c for c in nusax_val.columns
                        if c not in ("indonesian", "english")
                        and not c.startswith("Unnamed")]
        print(f"Dialect columns: {DIALECT_COLS}")

        preds_n, refs_n, N_PER = [], [], min(20, len(nusax_val))
        for d in DIALECT_COLS:
            for _, row in nusax_val[["indonesian", d]].dropna().head(N_PER).iterrows():
                inputs = norm_tok(
                    str(row[d]), return_tensors="pt",
                    truncation=True, max_length=256,
                ).to(DEVICE)
                with torch.no_grad():
                    out = norm_mod.generate(
                        **inputs, max_new_tokens=256,
                        num_beams=4, no_repeat_ngram_size=3, early_stopping=True,
                    )
                preds_n.append(norm_tok.decode(out[0], skip_special_tokens=True))
                refs_n.append(str(row["indonesian"]))

        bleu = sacrebleu.corpus_bleu(preds_n, [refs_n])
        chrf = sacrebleu.corpus_chrf(preds_n, [refs_n])
        norm_metrics = {"bleu4": round(bleu.score, 2),
                        "chrf": round(chrf.score, 2),
                        "n_samples": len(preds_n)}
        print(f"BLEU-4 = {norm_metrics['bleu4']:.2f}  (target > 20)")
        print(f"chrF++ = {norm_metrics['chrf']:.2f}  |  n = {norm_metrics['n_samples']}")
        free_vram(norm_mod, norm_tok)


## 12. DialectBridge — pipeline end-to-end

Menggabungkan Stage 1 (Normalizer) dan Stage 2 (Summarizer). QC gate: output normalisasi harus > 30% panjang input; jika tidak, teks dialek asli diteruskan langsung ke summarizer.


In [ ]:
class DialectBridge:
    """
    Two-stage dialect summarization pipeline.
    Stage 1: dialect text --> BI baku  (mT5-small normalizer)
    Stage 2: BI baku     --> BI summary (IndoT5 / IndoBART / mT5-base)
    """

    SUPPORTED = {
        "indot5":   (str(INDOT5_DIR),   "t5"),
        "indobart": (str(INDOBART_DIR), "bart"),
        "mt5base":  (str(MT5BASE_DIR),  "mt5"),
    }

    def __init__(self, summarizer="indot5"):
        if summarizer not in self.SUPPORTED:
            raise ValueError(f"Unsupported summarizer: {summarizer}")
        self.summarizer_name = summarizer
        self.norm_tok = AutoTokenizer.from_pretrained(str(NORM_DIR))
        self.norm_mod = AutoModelForSeq2SeqLM.from_pretrained(str(NORM_DIR)).to(DEVICE).eval()
        path, self.sum_type = self.SUPPORTED[summarizer]
        self.sum_tok = AutoTokenizer.from_pretrained(
            path, use_fast=False if self.sum_type == "bart" else True
        )
        self.sum_mod = AutoModelForSeq2SeqLM.from_pretrained(path).to(DEVICE).eval()

    def _normalize(self, text: str) -> str:
        inputs = self.norm_tok(
            text, return_tensors="pt", truncation=True, max_length=256
        ).to(DEVICE)
        with torch.no_grad():
            out = self.norm_mod.generate(
                **inputs, max_new_tokens=256,
                num_beams=4, no_repeat_ngram_size=3, early_stopping=True,
            )
        return self.norm_tok.decode(out[0], skip_special_tokens=True)

    def _summarize(self, text: str) -> str:
        t = preprocess_abstractive(text)
        if self.sum_type == "mt5":
            t = "summarize: " + t
        inputs = self.sum_tok(
            t, return_tensors="pt", truncation=True, max_length=512
        ).to(DEVICE)
        with torch.no_grad():
            out = self.sum_mod.generate(**inputs, **GEN_KWARGS)
        return self.sum_tok.decode(out[0], skip_special_tokens=True)

    def bridge(self, dialect_text: str, verbose: bool = True) -> dict:
        """Teks dialek --> ringkasan Bahasa Indonesia baku."""
        t0        = time.time()
        normalized = self._normalize(dialect_text)
        n_in      = len(dialect_text.split())
        qc_passed = (len(normalized.split()) / max(n_in, 1)) >= 0.30
        summary   = self._summarize(normalized if qc_passed else dialect_text)
        result = {
            "input_dialect"    : dialect_text,
            "normalized_bi"    : normalized,
            "qc_passed"        : bool(qc_passed),
            "summary"          : summary,
            "summarizer_used"  : self.summarizer_name,
            "compression_ratio": round(len(summary.split()) / max(n_in, 1), 4),
            "elapsed_sec"      : round(time.time() - t0, 2),
        }
        if verbose:
            print(f"[{self.summarizer_name}]  elapsed={result['elapsed_sec']}s  qc={qc_passed}")
            print(f"  INPUT      : {dialect_text[:120]}")
            print(f"  NORMALIZED : {normalized[:120]}")
            print(f"  SUMMARY    : {summary[:120]}")
        return result

    def free(self):
        free_vram(self.norm_mod, self.sum_mod, self.norm_tok, self.sum_tok)


## 13. Level 2 — robustness dialek

In [ ]:
def build_dialect_test_set(df_nusax, dialect_col, n_articles=30, n_sents=5):
    """Gabungkan kalimat NusaX menjadi pseudo-artikel untuk evaluasi robustness."""
    rows = (
        df_nusax[["indonesian", dialect_col]].dropna().reset_index(drop=True)
    )
    rows = rows[rows[dialect_col].astype(str).str.strip().astype(bool)]
    articles = []
    for start in range(0, min(n_articles * n_sents, len(rows)), n_sents):
        chunk = rows.iloc[start:start + n_sents]
        if len(chunk) < n_sents:
            break
        articles.append({
            "text_dialect"   : " ".join(str(x) for x in chunk[dialect_col]),
            "text_indonesian": " ".join(str(x) for x in chunk["indonesian"]),
            "summary"        : str(rows.iloc[start]["indonesian"]),
            "dialect"        : dialect_col,
        })
    return articles


dialect_test = []
local_dir    = ROOT / "dataset" / "nusax" / "datasets" / "mt"
if local_dir.exists():
    nusax_full = pd.concat(
        [pd.read_csv(local_dir / f)
         for f in ["train.csv", "valid.csv", "test.csv"]
         if (local_dir / f).exists()],
        ignore_index=True,
    )
    for d in ["javanese", "sundanese", "minangkabau"]:
        if d in nusax_full.columns:
            dialect_test.extend(
                build_dialect_test_set(nusax_full, d, n_articles=30, n_sents=5)
            )

print(f"Dialect test set: {len(dialect_test)} pseudo-articles")
if dialect_test:
    print(f"  Sample dialect : {dialect_test[0]['text_dialect'][:80]}")


In [ ]:
robustness = {}

if not dialect_test:
    print("SKIP — dialect test set kosong")
elif not NORM_DIR.exists():
    print("SKIP — normalizer tidak ditemukan")
else:
    _scorer = rs.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)

    def _agg(preds, refs):
        scores = [_scorer.score(r, p) for p, r in zip(preds, refs)]
        n = len(scores)
        return {k: round(sum(s[k].fmeasure for s in scores) / n, 4)
                for k in ("rouge1", "rouge2", "rougeL")}

    for summ_key in ["indot5", "indobart"]:
        if not (MODELS_DIR / summ_key).exists():
            print(f"SKIP {summ_key} — model tidak ditemukan")
            continue

        print(f"\n=== {summ_key} ===")
        bridge        = DialectBridge(summarizer=summ_key)
        refs_all      = [a["summary"] for a in dialect_test]
        preds_without = [bridge._summarize(a["text_dialect"]) for a in dialect_test]
        preds_with    = [
            bridge.bridge(a["text_dialect"], verbose=False)["summary"]
            for a in dialect_test
        ]

        r_without = _agg(preds_without, refs_all)
        r_with    = _agg(preds_with,    refs_all)

        per_dialect = {}
        for d in sorted(set(a["dialect"] for a in dialect_test)):
            idx = [i for i, a in enumerate(dialect_test) if a["dialect"] == d]
            w_  = _agg([preds_without[i] for i in idx], [refs_all[i] for i in idx])
            n_  = _agg([preds_with[i]    for i in idx], [refs_all[i] for i in idx])
            per_dialect[d] = {
                "without_norm" : w_,
                "with_norm"    : n_,
                "delta_rouge1" : round(n_["rouge1"] - w_["rouge1"], 4),
            }

        robustness[summ_key] = {
            "without_norm" : r_without,
            "with_norm"    : r_with,
            "delta_rouge1" : round(r_with["rouge1"] - r_without["rouge1"], 4),
            "per_dialect"  : per_dialect,
            "preds_without": preds_without,
            "preds_with"   : preds_with,
        }
        bridge.free()
        print(f"  Without norm : R1={r_without['rouge1']:.4f}")
        print(f"  With    norm : R1={r_with['rouge1']:.4f}")
        print(f"  Delta R1     : {robustness[summ_key]['delta_rouge1']:+.4f}")


## 14. Tabel hasil final

In [ ]:
SEP = "=" * 76

print(SEP)
print(f"Level 1 — Summarizer Evaluation  (IndoSum test, n={len(df_test)})")
print(SEP)
print(f"  {'Method':<12} {'ROUGE-1':>8} {'ROUGE-2':>8} {'ROUGE-L':>8}"
      f" {'BERT-F1':>10} {'CR':>7}")
print("-" * 76)
for row in results_rows:
    bs = f"{row['bertscore_f1']:.4f}" if row["bertscore_f1"] is not None else "     —"
    print(f"  {row['method']:<12} {row['rouge1']:>8.4f} {row['rouge2']:>8.4f}"
          f" {row['rougeL']:>8.4f} {bs:>10} {row['cr']:>7.4f}")

print()
print(SEP)
print("Normalizer — Stage 1  (NusaX validation)")
print(SEP)
print(f"  BLEU-4 : {norm_metrics.get('bleu4')}  (target > 20)")
print(f"  chrF++ : {norm_metrics.get('chrf')}")
print(f"  n      : {norm_metrics.get('n_samples')}")

if robustness:
    print()
    print(SEP)
    print("Level 2 — Dialect Robustness  (NusaX pseudo-articles)")
    print(SEP)
    print(f"  {'Summarizer':<14} {'Without Norm R1':>17}"
          f" {'With Norm R1':>14} {'Delta R1':>10}")
    print("-" * 76)
    for k, r in robustness.items():
        wn = f"{r['without_norm']['rouge1']:.4f}" if r.get("without_norm") else "—"
        wi = f"{r['with_norm']['rouge1']:.4f}"    if r.get("with_norm")    else "—"
        d  = f"{r['delta_rouge1']:+.4f}"           if "delta_rouge1" in r  else "—"
        print(f"  {k:<14} {wn:>17} {wi:>14} {d:>10}")
    print()
    print("  Delta > 0 = bukti empiris Two-Stage pipeline bekerja.")


## 15. Contoh konkret per metode (3 sampel)

In [ ]:
for idx in range(min(3, len(df_test))):
    art = df_test.iloc[idx]
    print("=" * 80)
    print(f"SAMPLE #{idx}  |  category={art['category']}")
    print("=" * 80)
    print(f"ARTICLE ({art['word_count']} words):")
    print(f"  {str(art['text'])[:300]}...")
    print()
    print(f"REFERENCE ({art['summary_word_count']} words):")
    print(f"  {art['summary'][:200]}")
    print()
    for method in ["TextRank", "NER", "IndoT5", "IndoBART", "mT5-base"]:
        preds = all_preds.get(method, [])
        if not preds:
            continue
        pred = preds[idx]
        print(f"[{method}] ({len(pred.split())} words):")
        print(f"  {pred[:200]}")
    print()


## 16. Demo DialectBridge — teks dialek arbitrer

Teks di bawah dibuat manual, bukan dari test set. Tidak ada data leakage.


In [ ]:
DEMO_INPUTS = [
    ("Kulo lapor dalan teng ngajeng griyo kulo rusak parah, sampun dangu mboten dipun beton. "
     "Warga ingkang langkung saking margi punika sring kacilakan motoripun.", "javanese"),
    ("Abdi ngalaporkeun yen jalan di payun bumi abdi parah pisan, atos lami teu diaspal. "
     "Warga anu ngalangkungan jalan eta sering kacilakaan motorna.", "sundanese"),
    ("Ambo malaporan jalan di muko rumah ambo rusak bana, alah lamo indak dibeton. "
     "Urang nan lewat di jalan ko sering kacilakaan motonyo.", "minangkabau"),
]

demo_outputs = []
if NORM_DIR.exists() and INDOT5_DIR.exists():
    bridge_demo = DialectBridge(summarizer="indot5")
    for text, dialect in DEMO_INPUTS:
        print(f"\n---- {dialect.upper()} ----")
        out = bridge_demo.bridge(text, verbose=True)
        out["dialect"] = dialect
        demo_outputs.append(out)
    bridge_demo.free()
else:
    print("SKIP — normalizer atau indot5 tidak ditemukan")


## 17. Simpan final_results.json

In [ ]:
final = {
    "metadata": {
        "n_test_samples": len(df_test),
        "models": {
            "normalizer": "google/mt5-small",
            "indot5"    : "cahya/t5-base-indonesian-summarization-cased",
            "indobart"  : "indobenchmark/indoBART",
            "mt5base"   : "google/mt5-base",
        },
    },
    "level1": {
        "summarizers": results_rows,
        "normalizer" : norm_metrics,
    },
    "level2_robustness": {
        k: {kk: vv for kk, vv in v.items()
            if kk not in ("preds_without", "preds_with")}
        for k, v in robustness.items()
    } if robustness else {},
    "demo_outputs": demo_outputs,
    "examples": [
        {
            "idx"              : i,
            "category"         : df_test.iloc[i]["category"],
            "reference_summary": df_test.iloc[i]["summary"],
            "predictions"      : {m: all_preds[m][i]
                                  for m in all_preds if all_preds[m]},
        }
        for i in range(min(3, len(df_test)))
    ],
}

out_path = OUTPUTS_DIR / "final_results.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(final, f, ensure_ascii=False, indent=2)
print(f"Saved -> {out_path}")
print(f"Size  = {out_path.stat().st_size / 1024:.1f} KB")


## Selesai

Output:
- `outputs/final_results.json` — semua metrik, prediksi, demo

**Membaca hasil:**
- L1 Summarizer: target ROUGE-1 > 0.30 untuk model abstractive, CR ~0.22
- Normalizer: target BLEU-4 > 20 di NusaX validation
- L2 Robustness: delta > 0 pada minimal 1 dialek = bukti Two-Stage pipeline bekerja
